# Critic-Agent trên Google Colab — model open-source 14B chạy bằng Ollama trên GPU T4

Notebook chạy Critic-Agent với một model mã nguồn mở lớn hơn nhiều so với `qwen2.5:3b` trên Mac (máy 8 GB không chạy nổi model 7B trở lên), hoàn toàn miễn phí: Ollama chạy trên GPU T4 (16 GB) của Colab, Critic-Agent gọi nó qua `localhost`. Sau đó chấm điểm (Meta-reviewer, thang 1–10 kiểu ICLR/NeurIPS) cho toàn bộ PDF trong thư mục Drive và so với điểm tham chiếu trong `eval/calibration.json`.

**Cách dùng**
1. Tải các PDF cần chấm lên Google Drive, vào thư mục `MyDrive/CO5151_Critic/papers` (giữ nguyên tên file để khớp với `eval/calibration.json`).
2. **Runtime → Change runtime type → T4 GPU**.
3. **Runtime → Run all**. Kết quả (`calibration_*.json`, `*.runs.jsonl`) và báo cáo từng bài (`runs/<run_id>/report.md`) được lưu vào `MyDrive/CO5151_Critic/`.
4. Nếu Colab bị ngắt: mở lại notebook và **Run all** lần nữa. Ollama và model phải tải lại (vài phút), nhưng các bài đã chấm xong được bỏ qua, chỉ chấm tiếp phần còn lại.

**Thời gian:** ô *Chạy thử một bài* đo thời gian thật cho một bài; nhân với số bài để ước lượng cả lượt chấm. Chưa đo trước trên T4, nên dựa vào số đo của ô đó.

**Chọn model (ô cấu hình):**
- `qwen2.5:14b` (mặc định, ~9 GB): cùng họ với model đang dùng nên định dạng JSON ổn định, không có chế độ "thinking".
- `qwen3:14b`: có thể mạnh hơn nhưng có chế độ thinking, chậm hơn; chưa thử với Critic-Agent, xem kết quả ô *Kiểm tra model* trước khi chạy cả lượt.
- Model ~20B trở lên thường không vừa 16 GB của T4 khi kèm context 16k.

In [ ]:
# Configuration
MODEL = "qwen2.5:14b"          # Ollama model tag; see the notes above
NUM_CTX = 16384                # context window; 16k + a 14B Q4 model fits a 16 GB T4 with one request at a time
SEEDS = "1"                    # e.g. "1 2 3" to measure how stable the score is (3x the time)
BRANCH = "feat/paper-score-domain-profiles"
DRIVE_DIR = "/content/drive/MyDrive/CO5151_Critic"

In [ ]:
# Check that a GPU is attached (should show a Tesla T4 or better)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Mount Google Drive: the papers are read from there and the results are written there, so they survive a disconnect
from google.colab import drive
drive.mount("/content/drive")

PAPERS_DIR = f"{DRIVE_DIR}/papers"
RUNS_DIR = f"{DRIVE_DIR}/runs"
OUT_DIR = f"{DRIVE_DIR}/results"
!mkdir -p "$PAPERS_DIR" "$RUNS_DIR" "$OUT_DIR"
!ls "$PAPERS_DIR" | head -20; echo "PDF count: $(ls "$PAPERS_DIR"/*.pdf 2>/dev/null | wc -l)"

In [ ]:
# Install Ollama and start the server in the background (one request at a time: parallel requests would each
# reserve a full NUM_CTX cache on the GPU; the 5 parallel critics simply queue)
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import os, subprocess, time, urllib.request
env = dict(os.environ, OLLAMA_NUM_PARALLEL="1", OLLAMA_KEEP_ALIVE="-1", OLLAMA_CONTEXT_LENGTH=str(NUM_CTX))
ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"),
                          stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        print("Ollama is up")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(open("/content/ollama.log").read()[-2000:])

In [ ]:
# Download the review model and the embedding model used by hybrid retrieval (a few minutes)
!ollama pull {MODEL} 2>&1 | tail -1
!ollama pull nomic-embed-text 2>&1 | tail -1
!ollama list

In [ ]:
# Model check: one JSON request like the ones Critic-Agent sends; prints the speed and whether the JSON parses
import json, requests
t = time.time()
r = requests.post("http://localhost:11434/api/chat", timeout=900, json={
    "model": MODEL, "stream": False, "format": "json",
    "messages": [{"role": "user", "content": "Give a 1-10 score for a paper that proposes RLHF for instruction "
                  'following. Return JSON: {"overall": <1-10>, "justification": <one sentence>}'}],
    "options": {"temperature": 0, "num_ctx": NUM_CTX, "num_predict": 300}}).json()
print(r["message"]["content"])
print(json.loads(r["message"]["content"]))
print(f"{r.get('eval_count', 0) / (r.get('eval_duration', 1) / 1e9):.1f} tokens/s generation, "
      f"{time.time() - t:.0f}s total (first call includes loading the model onto the GPU)")

In [ ]:
# Get the code and its pinned dependencies (uv installs Python 3.12 if the session has another version)
REPO = "/content/C05151_Next-generation_Software_Technology"
!if [ -d "$REPO" ]; then git -C "$REPO" fetch -q origin && git -C "$REPO" checkout -q "$BRANCH" && git -C "$REPO" pull -q; else git clone -q -b "$BRANCH" https://github.com/anho272202-HCMUT/C05151_Next-generation_Software_Technology.git "$REPO"; fi
%cd {REPO}
!git log --oneline -1
!pip install -q uv
!uv sync --frozen -q
# Offline tests with the mock LLM (run before .env exists, since .env would override the test settings)
!uv run pytest -q 2>&1 | tail -1

In [ ]:
# Settings for a 14B model with a 16k context: larger evidence budgets than the 3B defaults (1800 tokens / 4 hits /
# 4000 chars), small enough that the longest prompt (a critic with its evidence) stays well under NUM_CTX
settings = {
    "LLM_PROVIDER": "ollama", "LLM_MODEL": MODEL, "OLLAMA_BASE_URL": "http://localhost:11434",
    "EMBED_MODEL": "nomic-embed-text", "OLLAMA_NUM_CTX": NUM_CTX,
    "CRITIC_EVIDENCE_TOKENS": 4000, "CRITIC_SEARCH_K": 6, "CRITIC_FIRST_PAGES_CHARS": 10000,
    "CRITIC_GUARDRAIL": 1, "CRITIC_PAPER_SCORE": 1,
    "PAPERS_DIR": PAPERS_DIR, "RUNS_DIR": RUNS_DIR, "CACHE_DIR": "/content/critic_cache",
}
open(".env", "w").write("".join(f"{k}={v}\n" for k, v in settings.items()))
!cat .env

In [ ]:
# Run one review first (the shortest PDF) to check the whole pipeline and measure the time per paper
import glob
first = min(glob.glob(f"{PAPERS_DIR}/*.pdf"), key=os.path.getsize)
print("Paper:", os.path.basename(first))
t = time.time()
!uv run critic-agent review "{os.path.basename(first)}" --seed 1 < /dev/null
print(f"\n{(time.time() - t) / 60:.1f} min for one paper")

In [ ]:
# Score every PDF. Rerun this cell (or Run all) after a disconnect: finished (paper, seed) pairs are skipped.
!uv run python scripts/calibrate.py --papers-dir "$PAPERS_DIR" --seeds $SEEDS --out-dir "$OUT_DIR" < /dev/null

In [ ]:
# Results table; reports of every review are in RUNS_DIR/<run_id>/report.md
import pandas as pd
stem = "calibration_" + MODEL.replace(":", "-").replace("/", "-")
summary = json.load(open(f"{OUT_DIR}/{stem}.json"))
print({k: v for k, v in summary.items() if k != "rows"})
pd.DataFrame(summary["rows"])[["file", "domain", "mean", "std", "scores", "reference"]]